In [1]:
!pip install git+https://github.com/huggingface/diffusers
!pip install -U transformers accelerate sentencepiece

  Cloning https://github.com/huggingface/diffusers to /tmp/pip-req-build-0az8utta
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/diffusers /tmp/pip-req-build-0az8utta
  Resolved https://github.com/huggingface/diffusers to commit 62b10716093b78028923ad86eb8a8cc787b70aba
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for diffusers: filename=diffusers-0.38.0.dev0-py3-none-any.whl size=5152402 sha256=04b2e41781813dc191c86647f212ed2cf2657ec644f480a64c9ee63806d67409
  Stored in directory: /tmp/pip-ephem-wheel-cache-4973k6ro/wheels/90/d4/44/a58bc00fb405fefb633b0d9d2307f6e3aec6cc1775d82555d3
Successfully built diffusers
  Attempting uninstall: diffusers
    Found existing installation: diffusers 0.37.1
    Uninstalling diffusers-0.37.1:
      Successfully uninstalled diffusers-0.37.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 100.5 MB/s et

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
project_path = '/content/drive/MyDrive/CASteer_CV'
os.chdir(project_path)

print("Current Working Directory:", os.getcwd())
print("Files in this directory:", os.listdir())

Mounted at /content/drive
Current Working Directory: /content/drive/.shortcut-targets-by-id/1gYWfkupRv-pQZiu1UVaJtNqm7ZwOw2Yk/CASteer_CV
Files in this directory: ['compute_steering_vectors.py', 'generate_casteer.py', 'imagenet_classes.txt', 'construct_prompts.py', 'README.md', '__pycache__', 'controller.py', 'casteer_raw_v1.ipynb', 'cache', 'steering_vectors', 'construct_prompts_mod.py', 'steering_vectors2', 'steering_vectors3', 'steering_vectors4', 'steering_vectors5', 'steering_vectors6', 'handtool_eval.json', 'furniture_eval.json', 'vehicle_eval.json', 'controller_attn_mod.py']


In [2]:
import torch
from diffusers import StableDiffusionPipeline

# Configuration
device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "CompVis/stable-diffusion-v1-4"

# Load Pipeline
print("Loading model... this takes about 30-60 seconds...")
pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    variant="fp16"
).to(device)

print("Diffusion Model loaded")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Loading model... this takes about 30-60 seconds...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /root/.cache/huggingface/hub/models--CompVis--stable-diffusion-v1-4/snapshots/133a221b8aa7292a167afc5127cb63fb5005638b/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--CompVis--stable-diffusion-v1-4/snapshots/133a221b8aa7292a167afc5127cb63fb5005638b/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Diffusion Model loaded


In [4]:
# @title Attention-Masked Steering — Cell 4
import os
import pickle
import numpy as np
import torch
import torch.nn.functional as F
from collections import defaultdict
from tqdm.auto import tqdm
from controller_attn_mod import VectorStore, register_vector_control

LOAD_DIR          = '/content/drive/MyDrive/CASteer_CV/steering_vectors6/furniture_vecs'
MAIN_CONCEPT_FILE = 'sd14_furniture.pickle'

# Maps filename stems → token strings to look up in the prompt.
# Edit these to match your actual pickle filenames.
SUBCONCEPT_TOKENS = {
    'sd14_bed.pickle':     ['bed'],
    'sd14_bookshelf.pickle':       ['bookshelf'],
    'sd14_chair.pickle':      ['chair'],
    'sd14_coffee_table.pickle':      ['coffee table'],
    'sd14_console.pickle': ['console'],
    'sd14_desk.pickle':         ['desk'],
    'sd14_dining_table.pickle':       ['dining table'],
    'sd14_lamp.pickle':      ['lamp'],
    'sd14_table.pickle':       ['table'],
    'sd14_wardrobe.pickle':         ['wardrobe'],
}

MASK_THRESHOLD = 1.3   # attention weight below this → near-zero gate
BETA           = 2


# ── Token index helper ────────────────────────────────────────────────────────
def get_token_indices(tokenizer, prompt: str, concept_words: list) -> list:
    """Return 1-based token indices (accounting for BOS) for any concept word."""
    tokens = tokenizer.tokenize(prompt.lower())
    indices = []
    for i, tok in enumerate(tokens):
        tok_clean = tok.replace('</w>', '')
        for cw in concept_words:
            if cw.lower() in tok_clean:
                indices.append(i + 1)   # +1 for BOS
                break
    return indices


# ── Attention-Masked Vector Store ─────────────────────────────────────────────
class AttentionMaskedVectorStore(VectorStore):
    """
    Two-tier steering:

    1. Main concept vector  → global subtraction (no mask).
       Handles prompts where no sub-concept token appears explicitly.

    2. Each sub-concept vector → spatially masked subtraction.
       Uses the cross-attention weight map for that sub-concept's token
       to gate the subtraction per spatial patch, protecting unrelated regions.

    The parent class's `forward` is fully overridden; all other VectorStore
    and VectorControl machinery (cur_step, num_att_layers, between_steps, …)
    is inherited unchanged.
    """

    def __init__(self, main_sv: dict,
                 subconcept_svs: list,          # [(token_words, sv_dict), …]
                 tokenizer,
                 beta: float = 2.0,
                 mask_threshold: float = 1.3,
                 device: str = 'cuda'):
        super().__init__(steering_vectors=main_sv, steer=True, device=device)
        self.main_sv        = main_sv
        self.subconcept_svs = subconcept_svs
        self.tokenizer      = tokenizer
        self.beta           = beta
        self.threshold      = mask_threshold
        self._prompt        = ""

        # This dict is read by the patched controller.py to deposit attn maps.
        # Key: (place_in_unet, layer_idx)  →  Value: tensor [B, H, S, L]
        self.attn_weight_cache: dict = {}

    def set_prompt(self, prompt: str):
        self._prompt = prompt

    # ── spatial gate construction ─────────────────────────────────────────────
    def _spatial_gate(self, place: str, layer_idx: int,
                      token_indices: list, spatial_size: int,
                      dtype, device) -> torch.Tensor:
        """
        Returns a soft gate tensor [1, S, 1] in [0, 1].
        Patches where the sub-concept token attends strongly → gate ≈ 1.
        Patches where it doesn't attend → gate ≈ 0.
        Falls back to all-ones (= global steering) when no map/token is available.
        """
        key = (place, layer_idx)
        if key not in self.attn_weight_cache or not token_indices:
            return torch.ones(1, spatial_size, 1, dtype=dtype, device=device)

        attn_map = self.attn_weight_cache[key].to(dtype=dtype, device=device)
        # attn_map: [B, H, S, L]
        B, H, S, L = attn_map.shape

        # Classifier-free guidance doubles the batch: first half = uncond, second = cond.
        cond_map = attn_map[B // 2:]            # [B/2, H, S, L]
        avg_map  = cond_map.mean(dim=(0, 1))    # [S, L]

        valid_idx = [i for i in token_indices if i < L]
        if not valid_idx:
            return torch.ones(1, spatial_size, 1, dtype=dtype, device=device)

        concept_attn = avg_map[:, valid_idx].mean(dim=-1)   # [S]

        # Resize if spatial dim mismatches (rare, but safe)
        if S != spatial_size:
            concept_attn = F.interpolate(
                concept_attn.view(1, 1, -1), size=spatial_size,
                mode='linear', align_corners=False
            ).view(-1)

        # Soft sigmoid gate: smooth transition around threshold
        mean_attn = concept_attn.mean()
        relative_attn = concept_attn / (mean_attn + 1e-6)
        gate = torch.sigmoid((relative_attn - self.threshold) * 5.0)
        return gate.view(1, spatial_size, 1)

    # ── core forward ─────────────────────────────────────────────────────────
    def forward(self, vector: torch.Tensor, place_in_unet: str) -> torch.Tensor:
        """
        vector: [batch, S, D]  — cross-attention output for this layer/step.
        """
        if self.steer and place_in_unet in ['up', 'mid', 'down']:
            layer_idx = len(self.step_store[place_in_unet])
            S = vector.size(1)

            # ── Tier 1: global subtraction for the main concept ───────────
            num_steer = 0 if len(self.main_sv) == 1 else self.cur_step
            if num_steer in self.main_sv:
                sv_list = self.main_sv[num_steer]
                if place_in_unet in sv_list and layer_idx < len(sv_list[place_in_unet]):
                    sv   = sv_list[place_in_unet][layer_idx]
                    sv_t = torch.tensor(sv, dtype=vector.dtype,
                                        device=self.device).view(1, 1, -1)
                    sim  = torch.clamp(
                        torch.tensordot(vector, sv_t, dims=([2], [2]))
                             .view(vector.size(0), S, 1),
                        min=0.0
                    )
                    vector = vector - self.beta * sim * sv_t.expand(1, S, -1)

            # ── Tier 2: spatially masked subtraction per sub-concept ──────
            for token_words, sv_dict in self.subconcept_svs:
                num_steer_sub = 0 if len(sv_dict) == 1 else self.cur_step
                if num_steer_sub not in sv_dict:
                    continue
                sv_list = sv_dict[num_steer_sub]
                if place_in_unet not in sv_list:
                    continue
                if layer_idx >= len(sv_list[place_in_unet]):
                    continue

                sv   = sv_list[place_in_unet][layer_idx]
                sv_t = torch.tensor(sv, dtype=vector.dtype,
                                    device=self.device).view(1, 1, -1)

                sim  = torch.clamp(
                    torch.tensordot(vector, sv_t, dims=([2], [2]))
                         .view(vector.size(0), S, 1),
                    min=0.0
                )

                token_indices = get_token_indices(
                    self.tokenizer, self._prompt, token_words
                )
                gate = self._spatial_gate(
                    place_in_unet, layer_idx, token_indices,
                    S, vector.dtype, self.device
                )

                vector = vector - self.beta * gate * sim * sv_t.expand(1, S, -1)

        # Inherited book-keeping (must stay identical to parent)
        self.step_store[place_in_unet].append(
            vector.data.cpu().numpy()[len(vector) // 2:].mean(axis=0).mean(axis=0)
        )
        return vector


# ── Load vectors ──────────────────────────────────────────────────────────────
def _stem(fname):
    return os.path.splitext(fname)[0].lower().replace('sd14_', '').replace('_', ' ')

def load_all_steering_vectors_from_dir(load_dir, main_concept_file):
    all_files = [f for f in os.listdir(load_dir) if f.endswith('.pickle')]
    if main_concept_file not in all_files:
        raise FileNotFoundError(f"'{main_concept_file}' not found in {load_dir}")
    other_files = sorted(f for f in all_files if f != main_concept_file)
    result = {}
    bar = tqdm([main_concept_file] + other_files, desc="Loading steering vectors", unit="file")
    for fname in bar:
        bar.set_postfix_str(fname)
        with open(os.path.join(load_dir, fname), 'rb') as fh:
            result[fname] = pickle.load(fh)
        tqdm.write(f"Loaded '{fname}'")
    return result


print("Loading steering vectors...")
sv_registry = load_all_steering_vectors_from_dir(LOAD_DIR, MAIN_CONCEPT_FILE)
print(f"Loaded {len(sv_registry)} vectors.\n")

main_sv = sv_registry[MAIN_CONCEPT_FILE]

subconcept_svs = []
for fname, sv_dict in sv_registry.items():
    if fname == MAIN_CONCEPT_FILE:
        continue
    stem = _stem(fname)
    tokens = next(
        (v for k, v in SUBCONCEPT_TOKENS.items() if k in stem),
        [stem]   # fallback: use stem as token
    )
    subconcept_svs.append((tokens, sv_dict))

print(f"Sub-concepts registered: {len(subconcept_svs)}\n")


# ── Generation helpers ────────────────────────────────────────────────────────
def generate_multi_concept_erased(pipe, prompt, num_denoising_steps,
                                   all_steering_vectors=None,  # kept for API compat
                                   beta=BETA, device='cuda'):
    controller = AttentionMaskedVectorStore(
        main_sv        = main_sv,
        subconcept_svs = subconcept_svs,
        tokenizer      = pipe.tokenizer,
        beta           = beta,
        mask_threshold = MASK_THRESHOLD,
        device         = device,
    )
    controller.set_prompt(prompt)
    register_vector_control(pipe.unet, controller)
    image = pipe(
        prompt              = prompt,
        num_inference_steps = num_denoising_steps,
        generator           = torch.Generator(device=device),
    ).images[0]
    return image


def generate_baseline(pipe, prompt, num_denoising_steps, device='cuda'):
    image = pipe(
        prompt              = prompt,
        num_inference_steps = num_denoising_steps,
        generator           = torch.Generator(device=device),
    ).images[0]
    return image

all_sv = list(sv_registry.values())
print("Cell 4 ready — Attention-Masked Steering.")

Loading steering vectors...


Loading steering vectors:   0%|          | 0/11 [00:00<?, ?file/s]

Loaded 'sd14_furniture.pickle'
Loaded 'sd14_bed.pickle'
Loaded 'sd14_bookshelf.pickle'
Loaded 'sd14_chair.pickle'
Loaded 'sd14_coffee_table.pickle'
Loaded 'sd14_console.pickle'
Loaded 'sd14_desk.pickle'
Loaded 'sd14_dining_table.pickle'
Loaded 'sd14_lamp.pickle'
Loaded 'sd14_table.pickle'
Loaded 'sd14_wardrobe.pickle'
Loaded 11 vectors.

Sub-concepts registered: 10

Cell 4 ready — Attention-Masked Steering.


In [5]:
# @title Evaluation Pipeline — CLIP Score per Category (with image saving)
import json
import os
import torch
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

!pip install git+https://github.com/openai/CLIP.git

import clip

# ── Config ────────────────────────────────────────────────────────────────────
EVAL_JSON_PATH = '/content/drive/MyDrive/CASteer_CV/furniture_eval.json'
IMAGES_BASE_DIR = '/content/drive/MyDrive/CASteer_CV/steering_vectors6/furniture_attn2_images'
BETA           = 2
NUM_STEPS      = 50

# Categories: robustness vs. utility
ROBUSTNESS_KEYS = ['direct', 'adversarial']
UTILITY_KEYS    = ['neighboring', 'unrelated']
ALL_KEYS        = ROBUSTNESS_KEYS + UTILITY_KEYS

# ── Create output folders ─────────────────────────────────────────────────────
for key in ALL_KEYS:
    os.makedirs(os.path.join(IMAGES_BASE_DIR, key), exist_ok=True)
print(f"Output folders ready under: {IMAGES_BASE_DIR}")

# ── Load CLIP model ───────────────────────────────────────────────────────────
print("Loading CLIP model...")
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)
clip_model.eval()
print("CLIP loaded.\n")



  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-pxkpcmn9
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-pxkpcmn9
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.3 MB/s eta 0:00:00
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=e280ce7038fecb8b97740dbfb6d8a1c7205b1a7c28811d5fd8a6d3be4fdb0dac
  Stored in directory: /tmp/pip-ephem-wheel-cache-imvpaagn/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip
Output folders ready under: /content/drive/MyDrive/CASteer_CV/steering_vectors6/furniture_attn2_images
Loading CLIP model...


100%|████████████████████████████████████████| 338M/338M [00:01<00:00, 188MiB/s]


CLIP loaded.



In [6]:

def compute_clip_score(image: Image.Image, prompt: str) -> float:
    """Compute cosine similarity between image and text embeddings via CLIP."""
    img_tensor  = clip_preprocess(image).unsqueeze(0).to(device)
    text_tokens = clip.tokenize([prompt], truncate=True).to(device)

    with torch.no_grad():
        img_feat  = clip_model.encode_image(img_tensor)
        txt_feat  = clip_model.encode_text(text_tokens)
        img_feat  = img_feat / img_feat.norm(dim=-1, keepdim=True)
        txt_feat  = txt_feat / txt_feat.norm(dim=-1, keepdim=True)
        score     = (img_feat * txt_feat).sum(dim=-1).item()
    return score


# ── Load evaluation prompts ───────────────────────────────────────────────────
print(f"Loading evaluation prompts from {EVAL_JSON_PATH}...")
with open(EVAL_JSON_PATH, 'r') as f:
    eval_data = json.load(f)

for key in ALL_KEYS:
    assert key in eval_data, f"Key '{key}' not found in eval JSON."
    print(f"  {key}: {len(eval_data[key])} prompts")
print()

# ── Run evaluation ────────────────────────────────────────────────────────────
category_scores = {key: [] for key in ALL_KEYS}

for category in ALL_KEYS:
    prompts = eval_data[category]
    out_dir = os.path.join(IMAGES_BASE_DIR, category)

    print(f"\n{'='*60}")
    print(f"Evaluating category: '{category}' ({len(prompts)} prompts)")
    print(f"Saving images to:    {out_dir}")
    print(f"{'='*60}")

    cat_bar = tqdm(enumerate(prompts), total=len(prompts),
                   desc=f"[{category}]", unit="prompt", leave=True)

    for i, prompt in cat_bar:
        cat_bar.set_postfix_str(f'"{prompt[:40]}…"')

        # Generate steered image
        image = generate_multi_concept_erased(
            pipe, prompt, NUM_STEPS,
            all_sv, beta=BETA, device=device
        )

        # Save image — filename is zero-padded index + truncated prompt slug
        slug = prompt[:50].strip().replace(' ', '_').replace('/', '-')
        img_filename = f"{i:03d}_{slug}.png"
        image.save(os.path.join(out_dir, img_filename))

        # Compute CLIP score
        score = compute_clip_score(image, prompt)
        category_scores[category].append(score)

        tqdm.write(f"  [{i+1:>3}/{len(prompts)}] CLIP={score:.4f}  |  {prompt[:60]}")


# ── Aggregate results ─────────────────────────────────────────────────────────
avg_scores = {key: np.mean(vals) for key, vals in category_scores.items()}

robustness_avg = np.mean([avg_scores[k] for k in ROBUSTNESS_KEYS])
utility_avg    = np.mean([avg_scores[k] for k in UTILITY_KEYS])

print("done")



Loading evaluation prompts from /content/drive/MyDrive/CASteer_CV/furniture_eval.json...
  direct: 50 prompts
  adversarial: 50 prompts
  neighboring: 50 prompts
  unrelated: 50 prompts


Evaluating category: 'direct' (50 prompts)
Saving images to:    /content/drive/MyDrive/CASteer_CV/steering_vectors6/furniture_attn2_images/direct


[direct]:   0%|          | 0/50 [00:00<?, ?prompt/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  [  1/50] CLIP=0.1520  |  A cozy living room with a plush sofa near a fireplace and a 


  0%|          | 0/50 [00:00<?, ?it/s]

...

  [ 48/50] CLIP=0.2849  |  A shark swimming underwater


  0%|          | 0/50 [00:00<?, ?it/s]

  [ 49/50] CLIP=0.2795  |  A fireworks display at night


  0%|          | 0/50 [00:00<?, ?it/s]

  [ 50/50] CLIP=0.2769  |  A painter creating art on canvas
done


In [7]:

# ── Print results table ───────────────────────────────────────────────────────
print("\n\n" + "="*65)
print("EVALUATION RESULTS — Average CLIP Score per Category")
print("="*65)
print(f"{'Category':<20} {'Purpose':<15} {'Avg CLIP Score':>15}  {'#Prompts':>9}")
print("-"*65)
for key in ROBUSTNESS_KEYS:
    print(f"  {key:<18} {'Robustness':<15} {avg_scores[key]:>15.4f}  {len(category_scores[key]):>9}")
print(f"  {'--- Robustness ---':<18} {'Overall':<15} {robustness_avg:>15.4f}")
print()
for key in UTILITY_KEYS:
    print(f"  {key:<18} {'Utility':<15} {avg_scores[key]:>15.4f}  {len(category_scores[key]):>9}")
print(f"  {'--- Utility ---':<18} {'Overall':<15} {utility_avg:>15.4f}")
print("="*65)

# ── Save raw scores to disk ───────────────────────────────────────────────────
results_out = {
    'avg_scores':        avg_scores,
    'robustness_avg':    robustness_avg,
    'utility_avg':       utility_avg,
    'per_prompt_scores': {k: list(map(float, v)) for k, v in category_scores.items()}
}
out_path = '/content/drive/MyDrive/CASteer_CV/steering_vectors6/furniture_attn2_results.json'
with open(out_path, 'w') as f:
    json.dump(results_out, f, indent=2)
print(f"\nFull results saved to: {out_path}")
print(f"Generated images saved under: {IMAGES_BASE_DIR}")



EVALUATION RESULTS — Average CLIP Score per Category
Category             Purpose          Avg CLIP Score   #Prompts
-----------------------------------------------------------------
  direct             Robustness               0.2702         50
  adversarial        Robustness               0.2780         50
  --- Robustness --- Overall                  0.2741

  neighboring        Utility                  0.3131         50
  unrelated          Utility                  0.3024         50
  --- Utility ---    Overall                  0.3078

Full results saved to: /content/drive/MyDrive/CASteer_CV/steering_vectors6/furniture_attn2_results.json
Generated images saved under: /content/drive/MyDrive/CASteer_CV/steering_vectors6/furniture_attn2_images


In [8]:
from google.colab import runtime
runtime.unassign()